# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Зайцева Алёна Александровна
**Группа:** ИНФА-221
**Вариант:** 4 (предметная область: Фармацевтика и клинические исследования)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [ ]:
!pip install -q Levenshtein pandas lxml

In [ ]:
from __future__ import annotations

import csv
import os
import re
import unicodedata
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import pandas as pd

# --- XML backend ---
try:
    from lxml import etree as ET
    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET
    LXML_AVAILABLE = False

# --- Levenshtein backend ---
try:
    import Levenshtein as _lev
    def edit_distance(a: str, b: str) -> int:
        return _lev.distance(a, b)
    LEV_BACKEND = "Levenshtein (C)"
except ImportError:
    def edit_distance(a: str, b: str) -> int:
        if a == b: return 0
        if not a: return len(b)
        if not b: return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]
    LEV_BACKEND = "pure-python fallback"

XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
WORK_DIR = os.getcwd()

def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad

pd.set_option("display.max_colwidth", 46)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print(f"XML backend        : {'lxml' if LXML_AVAILABLE else 'xml.etree.ElementTree'}")
print(f"Levenshtein backend: {LEV_BACKEND}")
print(f"pandas             : {pd.__version__}")
print(f"Рабочий каталог    : {WORK_DIR}")

XML backend        : lxml
Levenshtein backend: Levenshtein (C)
pandas             : 2.2.3
Рабочий каталог    : /content


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [ ]:
TERMS: List[Dict[str, str]] = [
    {"en": "Sarcopenia", "ru": "Саркопения", "pos": "Noun", "def_en": "A progressive and generalized skeletal muscle disorder (or disease)."},
    {"en": "Screening", "ru": "Скрининг", "pos": "Noun", "def_en": "Identification of risk using questionnaires or tools before full diagnosis."},
    {"en": "Muscle mass", "ru": "Мышечная масса", "pos": "Noun phrase", "def_en": "A component of body composition evaluated via instrumental methods (DXA, BIA, CT, MRI)."},
    {"en": "Muscle strength", "ru": "Мышечная сила", "pos": "Noun phrase", "def_en": "The primary diagnostic criterion for sarcopenia, often assessed via dynamometry."},
    {"en": "Handgrip dynamometry", "ru": "Кистевая динамометрия", "pos": "Noun phrase", "def_en": "A simple, inexpensive, and non-invasive method for assessing muscle strength."},
    {"en": "Physical performance", "ru": "Физическая работоспособность", "pos": "Noun phrase", "def_en": "Functionality assessed by tests (gait speed, chair stand) used as an indicator of disease severity."},
    {"en": "Bioelectrical impedance analysis (BIA)", "ru": "Биоимпедансный анализ (БИА)", "pos": "Noun phrase", "def_en": "An accessible method for assessing body composition based on the electrical resistance of tissues."},
    {"en": "Dual-energy X-ray absorptiometry (DXA)", "ru": "Двухэнергетическая рентгеновская абсорбциометрия (DXA)", "pos": "Noun phrase", "def_en": "A key method for assessing muscle mass based on differential X-ray absorption by soft tissues."},
    {"en": "Cutoff values", "ru": "Пороговые значения", "pos": "Noun phrase", "def_en": "Standardized threshold limits used to define low muscle mass or strength."},
    {"en": "Frailty", "ru": "Старческая астения", "pos": "Noun", "def_en": "A geriatric phenotype/syndrome in which weakness is a key marker of vulnerability."},
    {"en": "Appendicular skeletal muscle mass index (ASMI)", "ru": "Индекс аппендикулярной мышечной массы (ИАММ)", "pos": "Noun phrase", "def_en": "A parameter calculated based on appendicular muscle mass divided by height."},
    {"en": "SARC-F", "ru": "SARC-F", "pos": "Acronym", "def_en": "A screening questionnaire assessing strength, assistance in walking, rise from a chair, climb stairs, falls."},
    {"en": "Supraependymal nerve plexus", "ru": "Супраэпендимное нервное сплетение", "pos": "Noun phrase", "def_en": "The neural plexus located on the ependyma's apical surface of the cerebral ventricles."},
    {"en": "Ependyma", "ru": "Эпендима", "pos": "Noun", "def_en": "The tissue lining the cerebral ventricles (described in the text as having an 'apical surface' and cells)."},
    {"en": "Immunohistochemistry", "ru": "Иммуногистохимия", "pos": "Noun", "def_en": "A method used to investigate signs of functional activity and identify fibers and structures exhibiting potential synaptic activity."},
    {"en": "Synaptophysin (SY)", "ru": "Синаптофизин (СФ)", "pos": "Noun", "def_en": "One of the proteins of the SNARE complex, acting as a regulator of neurotransmitter exocytosis from synaptic vesicles."},
    {"en": "Tryptophan hydroxylase (TPH)", "ru": "Триптофангидроксилаза (ТПГ)", "pos": "Noun phrase", "def_en": "The enzyme that limits the synthesis of serotonin."},
    {"en": "Tyrosine hydroxylase (TH)", "ru": "Тирозингидроксилаза (ТГ)", "pos": "Noun phrase", "def_en": "The enzyme that limits the synthesis of catecholamines."},
    {"en": "Cerebrospinal fluid (CSF)", "ru": "Спинномозговая жидкость (СМЖ)", "pos": "Noun phrase", "def_en": "A fluid whose biochemical composition serves as an important diagnostic criterion for a number of diseases."},
    {"en": "Catecholamines", "ru": "Катехоламины", "pos": "Noun", "def_en": "A group of neurotransmitters whose synthesis is limited by tyrosine hydroxylase."},
    {"en": "Serotonin", "ru": "Серотонин", "pos": "Noun", "def_en": "A neurotransmitter whose synthesis is limited by tryptophan hydroxylase."},
    {"en": "Synaptic activity", "ru": "Синаптическая активность", "pos": "Noun phrase", "def_en": "Activity manifested by the appearance of synaptophysin in synaptic terminals, which is traditionally considered a manifestation of synaptic activity by the cell."},
    {"en": "Compensated advanced chronic liver disease (cACLD)", "ru": "Компенсированное продвинутое хроническое заболевание печени (КПХЭП)", "pos": "Noun phrase", "def_en": "It represents severe fibrosis and compensated cirrhosis as a continuum, the stages of which are often impossible to distinguish in clinical practice."},
    {"en": "Metabolic dysfunction-associated steatoic liver disease (MASLD)", "ru": "Метаболически ассоциированная жировая болезнь печени (МАЖБП)", "pos": "Noun phrase", "def_en": "The presence of liver steatosis in combination with at least one of the five cardiometabolic risk factors."},
    {"en": "Arterial hypertension (AH)", "ru": "Артериальная гипертензия (АГ)", "pos": "Noun phrase", "def_en": "A cardiometabolic risk factor studied in the groups."},
    {"en": "Type 2 diabetes mellitus (T2DM)", "ru": "Сахарный диабет 2-го типа (СД2)", "pos": "Noun phrase", "def_en": "A determining factor for the development of cACLD in patients with MASLD."},
    {"en": "Dyslipidemia", "ru": "Дислипидемия", "pos": "Noun", "def_en": "Defined based on presence of one or more of the following criteria: triglyceride concentration ≥1.7 mmol/l, high-density lipoprotein concentration <1.0 mmol/l (in men) and <1.3 mmol/l (in women), or taking lipid-lowering therapy."},
    {"en": "Transient elastography (TE)", "ru": "Транзиентная эластография (ТЭ)", "pos": "Noun phrase", "def_en": "Non-invasive method for assessing fibrosis to identify patients with a likely progressive course of the disease."},
    {"en": "Controlled attenuation parameter (CAP)", "ru": "Контролируемый параметр затухания ультразвука (КПЭУ)", "pos": "Noun phrase", "def_en": "A parameter used to assess the severity of steatosis."},
    {"en": "Odds ratio (OR)", "ru": "Отношение шансов (ОШ)", "pos": "Noun phrase", "def_en": "A measure calculated with the boundaries defined at the 95% confidence interval to quantitatively assess the effect."},
    {"en": "Logistic regression", "ru": "Логистическая регрессия", "pos": "Noun phrase", "def_en": "A method used for the construction of a prognostic model for the risk of developing cACLD."},
    {"en": "Waist circumference", "ru": "Окружность талии", "pos": "Noun phrase", "def_en": "An anthropometric parameter that is a determining factor for the development of cACLD."},
    {"en": "PNPLA3 I148M", "ru": "PNPLA3 I148M", "pos": "Noun phrase", "def_en": "Patatin-Like Phospholipase Domain Containing 3 gene polymorphism."},
    {"en": "Non-alcoholic fatty liver disease (NAFLD)", "ru": "Неалкогольная жировая болезнь печени (НАЖБП)", "pos": "Noun phrase", "def_en": "The most common liver disease worldwide and the leading cause of liver-related morbidity and mortality."},
    {"en": "Hepatocellular carcinoma (HCC)", "ru": "Гепатоцеллюлярный рак (ГЦР)", "pos": "Noun phrase", "def_en": "An unfavorable outcome of MASLD progression, alongside liver cirrhosis."},
    {"en": "Liver stiffness", "ru": "Жесткость печени", "pos": "Noun phrase", "def_en": "A parameter measured by TE, with threshold values proposed to identify cACLD."},
    {"en": "Steatosis", "ru": "Стеатоз", "pos": "Noun", "def_en": "Fat accumulation in the liver, whose severity is assessed by the CAP function."},
    {"en": "Descriptive statistics", "ru": "Описательная статистика", "pos": "Noun phrase", "def_en": "A branch of statistics devoted to the summarization and description of data, which provides tools to explore data sets and illustrate the data."},
    {"en": "Qualitative variables", "ru": "Качественные переменные", "pos": "Noun phrase", "def_en": "Variables that cannot be measured, instead they are classified as categories."},
    {"en": "Quantitative variables", "ru": "Количественные переменные", "pos": "Noun phrase", "def_en": "Variables that can be measured numerically, or in other words their magnitude can be represented by a metric scale."},
    {"en": "General population", "ru": "Генеральная совокупность", "pos": "Noun phrase", "def_en": "The entire group of individuals or items from which a sample is drawn (implied by the text comparing it to random samples)."},
    {"en": "Random sample", "ru": "Случайная выборка", "pos": "Noun phrase", "def_en": "A subset of individuals chosen from a larger set (population) randomly."},
    {"en": "Mean (arithmetic mean)", "ru": "Среднее арифметическое", "pos": "Noun phrase", "def_en": "A sum of all values, divided by their total number (n)."},
    {"en": "Median", "ru": "Медиана", "pos": "Noun", "def_en": "The 0.5 quantile, which divides the distribution into two equal intervals; 50% of all values are located below the median and another 50% above."},
    {"en": "Mode", "ru": "Мода", "pos": "Noun", "def_en": "The most frequent value of the studied variable (or the absolute maximum of the distribution - the highest point on the histogram)."},
    {"en": "Standard deviation (SD)", "ru": "Стандартное отклонение (среднеквадратическое отклонение)", "pos": "Noun phrase", "def_en": "A parameter that describes the spread of the data, calculated as the square root of the variance."},
    {"en": "Confidence interval (CI)", "ru": "Доверительный интервал (ДИ)", "pos": "Noun phrase", "def_en": "An interval estimate which provides a range of possible values that likely includes the true value of the parameter for the general population."},
    {"en": "Histogram", "ru": "Гистограмма", "pos": "Noun", "def_en": "A graphical representation of the data distribution method where observed values are grouped into small ranges (bins) and counted."},
    {"en": "Interquartile range (IQR)", "ru": "Интерквартильный размах (ИКР)", "pos": "Noun phrase", "def_en": "The difference between the lower and the upper quartiles, providing information about the spread of the middle 50% of the data."},
    {"en": "Box-plot (box-and-whiskers diagram)", "ru": "Диаграмма размаха («ящик с усами», «бокс-плот»)", "pos": "Noun", "def_en": "A diagram that graphically represents quartiles (including median), as well as minimum and maximum values."},
    {"en": "Outliers", "ru": "Выбросы", "pos": "Noun", "def_en": "Data points that significantly differ from other observations."},
    {"en": "Odds", "ru": "Шансы", "pos": "Noun", "def_en": "The ratio between the probability that the outcome will occur and the probability that the outcome will not occur."},
    {"en": "Probability density function", "ru": "Функция плотности вероятностей", "pos": "Noun phrase", "def_en": "A function demonstrated by normalizing the histogram (dividing the count in each column by the number of observations multiplied by the class width)."},
    {"en": "Normal distribution (Gaussian)", "ru": "Нормальное распределение (гауссово)", "pos": "Noun phrase", "def_en": "A probability distribution that is symmetric about the mean (a bell-shaped curve), or in other words, the closer the value is to the mean, the higher is the frequency."},
    {"en": "Skewness", "ru": "Коэффициент асимметрии", "pos": "Noun", "def_en": "Measure of asymmetry of the distribution."},
    {"en": "Kurtosis", "ru": "Коэффициент эксцесса", "pos": "Noun", "def_en": "Measure of the combined weight of a distribution's tails relative to the center of the distribution."},
    {"en": "Studies and research design", "ru": "Дизайн научных исследований", "pos": "Noun phrase", "def_en": "Adequate design is an essential condition for conducting a successful study, a concept which is often underestimated and overlooked."},
    {"en": "Subject", "ru": "Субъект", "pos": "Noun", "def_en": "An individual (patient) participating in the study."},
    {"en": "Prospective studies", "ru": "Проспективные исследования", "pos": "Noun phrase", "def_en": "In prospective studies a group of subjects is actively followed by the researchers for predefined period of time to determine the outcomes that will happen in the future."},
    {"en": "Retrospective studies", "ru": "Ретроспективные исследования", "pos": "Noun phrase", "def_en": "In retrospective studies, the authors have information about existing outcomes and collect data on past exposures (e.g., from medical records)."},
    {"en": "Prevalence", "ru": "Показатель распространенности (превалентность)", "pos": "Noun", "def_en": "The proportion of a population who have a specific characteristic at a specific time point or in a given time period, regardless of when they first developed the characteristic."},
    {"en": "Incidence", "ru": "Показатель заболеваемости (инцидентность)", "pos": "Noun", "def_en": "Incidence indicates the number of new cases of a disease or condition that develop in a population in a specified time period."},
    {"en": "Association", "ru": "Ассоциация", "pos": "Noun", "def_en": "Association is a statistical relationship between two variables (e.g., exposure and outcome), however it does not necessarily mean that there is a cause-effect relationship between them."},
    {"en": "Causation", "ru": "Причинно-следственная связь", "pos": "Noun", "def_en": "Causation means that the exposure produces the effect."},
    {"en": "Bias", "ru": "Систематическая ошибка", "pos": "Noun", "def_en": "Bias is a systematic error in the interpretation of the data due to a factor that has not been accounted for in statistical analysis."},
    {"en": "Confounding", "ru": "Конфаундинг", "pos": "Noun", "def_en": "Confounding is a distortion of the true relationship between exposure and outcome by the influence of one or more factors, called confounders."},
    {"en": "Selection bias", "ru": "Систематическая ошибка отбора", "pos": "Noun phrase", "def_en": "Selection bias occurs when systematic differences exist between baseline characteristics of the groups that are compared."},
    {"en": "Recall bias", "ru": "Систематическая ошибка памяти", "pos": "Noun phrase", "def_en": "Recall bias occurs in retrospective studies when participants do not remember previous events or experiences accurately or omit details."},
    {"en": "Observer bias", "ru": "Систематическая ошибка наблюдателя", "pos": "Noun phrase", "def_en": "Observer bias occurs if the investigator knows the exposure status of the subject – this knowledge can influence measurements (in other words the researcher sees what he wants to see)."},
    {"en": "Attrition bias", "ru": "Систематическая ошибка выбывания", "pos": "Noun phrase", "def_en": "Attrition bias occurs when participants leave during a study."},
    {"en": "Publication bias", "ru": "Систематическая ошибка публикации", "pos": "Noun phrase", "def_en": "Publication bias occurs when the outcome of a study influences the decision whether to publish it."},
    {"en": "Basic studies", "ru": "Фундаментальные исследования", "pos": "Noun phrase", "def_en": "Basic studies, also known as experimental research, are designed to assess cause-outcome relationships between the variation of the independent variable and its effect on dependent variable in a highly controlled setting."},
    {"en": "Observational studies", "ru": "Наблюдательные исследования", "pos": "Noun phrase", "def_en": "In observational studies, the researcher does not interfere with the course of events and does not influence the participants and the factors being studied."},
    {"en": "Cross-sectional studies", "ru": "Поперечные (одномоментные) исследования", "pos": "Noun phrase", "def_en": "Cross-sectional studies (surveys) are used to assess the prevalence of diseases, risk factors, outcomes and other health-related characteristics in a particular population in a certain moment of time."},
    {"en": "Ecological studies", "ru": "Экологические исследования", "pos": "Noun phrase", "def_en": "Ecological (correlation) studies investigate associations between disease occurrence and exposure to potential risk factor."},
    {"en": "Case-control studies", "ru": "Исследования «случай-контроль»", "pos": "Noun phrase", "def_en": "Case-control studies are conducted retrospectively and analyse participants identified on the basis of their case status, i.e., presence or absence of the disease or another outcome."},
    {"en": "Cohort studies", "ru": "Когортные исследования", "pos": "Noun phrase", "def_en": "In cohort studies the researchers identify study participants based on their exposure status and either follow them over time (prospective setting) to identify which participants will develop the outcome, or looking back at data created in the past (retrospective setting) prior to the development of the outcome."},
    {"en": "Interventional studies", "ru": "Экспериментальные (интервенционные) исследования", "pos": "Noun phrase", "def_en": "Interventional (experimental) studies compare the effect of the studied treatment (intervention) in the experimental group with a control group of subjects, who receive either placebo or a different treatment."},
    {"en": "Randomised controlled trials (RCT)", "ru": "Рандомизированные контролируемые исследования (РКИ)", "pos": "Noun phrase", "def_en": "In randomised controlled trials (RCT) a homogenous group of subjects is randomly (by chance) divided into two separate groups."},
    {"en": "Systematic review and meta-analysis", "ru": "Систематический обзор и метаанализ", "pos": "Noun phrase", "def_en": "Systematic reviews evaluate and interpret the results of all published studies in the clinical area... statistical analysis to combine the results of separate studies, included in a systematic review, this method is called meta-analysis."},
    {"en": "COVID-19 (Corona Virus Disease 2019)", "ru": "Новая коронавирусная инфекция (COVID-19)", "pos": "Noun", "def_en": "An infectious disease caused by the SARS-CoV-2 virus, which became a major issue of Public Health."},
    {"en": "SARS-CoV-2 (severe acute respiratory syndrome coronavirus 2)", "ru": "Коронавирус тяжелого острого респираторного синдрома – 2 (SARS-CoV-2)", "pos": "Noun", "def_en": "An enveloped, non-segmented positive-sense RNA belonging to the Nidovirales order."},
    {"en": "Pregnancy", "ru": "Беременность", "pos": "Noun", "def_en": "A condition potentially associated with a higher burden of maternal mortality and morbidity due to peculiar cardiovascular and respiratory maternal adaptations."},
    {"en": "Retrospective cohort study", "ru": "Ретроспективное когортное исследование", "pos": "Noun phrase", "def_en": "An international study design that looks back at data created in the past to analyze participants identified on the basis of their case status."},
    {"en": "Symptomatic", "ru": "С симптомами (симптомный)", "pos": "Adjective", "def_en": "Showing symptoms of the disease, such as cough and fever, followed by shortness of breath."},
    {"en": "Asymptomatic", "ru": "Бессимптомный", "pos": "Adjective", "def_en": "Not showing symptoms of the disease at the time of triage."},
    {"en": "Maternal outcomes", "ru": "Материнские показатели (исходы)", "pos": "Noun phrase", "def_en": "The presence of maternal mortality and morbidity including at least one of the following: admission to ICU, use of mechanical ventilation, or death."},
    {"en": "Neonatal outcomes", "ru": "Неонатальные показатели", "pos": "Noun phrase", "def_en": "Health outcomes related to the newborn, including admission rates to neonatal intensive care unit and neonatal death."},
    {"en": "Intensive care unit (ICU)", "ru": "Отделение интенсивной терапии (ОИТ)", "pos": "Noun phrase", "def_en": "A hospital department where patients with severe respiratory symptoms or critical conditions are admitted."},
    {"en": "Mechanical ventilation", "ru": "Механическая вентиляция легких", "pos": "Noun phrase", "def_en": "The use of a machine to assist or replace spontaneous breathing in patients with severe respiratory failure."},
    {"en": "Extracorporeal membrane oxygenation (ECMO)", "ru": "Экстракорпоральная мембранная оксигенация (ЭКМО)", "pos": "Noun phrase", "def_en": "A life support technique used in cases of severe respiratory or cardiac failure."},
    {"en": "Cesarean delivery", "ru": "Кесарево сечение", "pos": "Noun phrase", "def_en": "A surgical procedure used to deliver a baby through incisions in the mother's abdomen and uterus."},
    {"en": "Preterm birth", "ru": "Преждевременные роды", "pos": "Noun phrase", "def_en": "Birth of a baby before the 37th week of gestation."},
    {"en": "Intrauterine growth restriction (IUGR)", "ru": "Задержка внутриутробного развития (ЗВУР)", "pos": "Noun phrase", "def_en": "Poor growth of a fetus while in the mother's womb during pregnancy."},
    {"en": "Low birth weight", "ru": "Низкая масса тела при рождении", "pos": "Noun phrase", "def_en": "A weight at birth that is lower than normal, often associated with premature birth."},
    {"en": "Neonatal intensive care unit", "ru": "Отделение интенсивной терапии новорожденных", "pos": "Noun phrase", "def_en": "A unit that provides intensive care for ill or premature newborn infants."},
    {"en": "Perinatal deaths", "ru": "Перинатальные потери (смертность)", "pos": "Noun phrase", "def_en": "Deaths that occur around the time of birth, including stillbirths and neonatal deaths."},
    {"en": "First trimester", "ru": "Первый триместр", "pos": "Noun phrase", "def_en": "The first three months of pregnancy, defined in the text as a period when infection is associated with a higher incidence of composite adverse fetal outcome."},
    {"en": "Stillbirth", "ru": "Мертворождение", "pos": "Noun", "def_en": "The birth of an infant that has died in the womb."},
    {"en": "Gestational age", "ru": "Срок беременности (гестационный возраст)", "pos": "Noun phrase", "def_en": "A measure of the age of a pregnancy, which is calculated from the first day of the woman's last menstrual period."}
]

SUBJECT_FIELD = "Фармацевтика и клинические исследования"
TBX_PATH = "variant_4.tbx"

def build_tbx(terms: Sequence[Dict[str, str]], path: str, subject_field: str = SUBJECT_FIELD) -> str:
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})
    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = f"Lab 01, variant 4, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"
    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field
        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]
        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path

build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")

Создан variant_4.tbx: 80794 байт, терминов: 100



## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [ ]:
import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple
import pandas as pd

try:
    from lxml import etree as ET
except ImportError:
    import xml.etree.ElementTree as ET

try:
    import Levenshtein as _lev
    def edit_distance(a: str, b: str) -> int:
        return _lev.distance(a, b)
except ImportError:
    def edit_distance(a: str, b: str) -> int:
        if a == b: return 0
        if not a: return len(b)
        if not b: return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]

XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"


def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл не найден: {file_path}")

    tree = ET.parse(file_path)
    root = tree.getroot()
    rows = []

    for entry in root.iter("termEntry"):
        entry_id = entry.get("id", "")
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            if not lang:
                continue

            terms = [t.text.strip() for t in lang_set.iter("term") if t.text and t.text.strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": note.text.strip() if note is not None and note.text else "",
                "definition": definition.text.strip() if definition is not None and definition.text else ""
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry_id,
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject
            })

    return pd.DataFrame(rows, columns=["entry_id", "source_term", "target_term", "pos", "definition", "subject_field"])


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл не найден: {file_path}")

    tree = ET.parse(file_path)
    root = tree.getroot()
    rows = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or "").split("-")[0].lower()
            if not lang:
                continue

            seg = tuv.find("seg")
            if seg is not None:
                text = "".join(seg.itertext()).strip()
                if text:
                    segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": domain_node.text.strip() if domain_node is not None and domain_node.text else "",
            "src_len": len(segments[src_lang])
        })

    frame = pd.DataFrame(rows, columns=["tuid", "source_segment", "target_segment", "domain", "src_len"])
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame, threshold: float = 70.0, context_before: Optional[str] = None, top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос."""
    if tm_df.empty:
        return []

    results = []
    for _, row in tm_df.iterrows():
        lev = levenshtein_ratio(normalize(source_query), normalize(row["source_segment"]))
        tsr = token_sort_ratio(source_query, row["source_segment"])
        score = max(lev, tsr)

        if score < threshold:
            continue

        context_matched = bool(context_before is not None and normalize(context_before) == normalize(str(row.get("prev_source", ""))))

        if score >= 100.0:
            match_type = "ICE (101%)" if context_matched else "Exact (100%)"
        elif score >= 95.0:
            match_type = "High Fuzzy"
        elif score >= 75.0:
            match_type = "Fuzzy"
        else:
            match_type = "Low Fuzzy"

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": match_type
        })

    results.sort(key=lambda x: (x["score"], x["tuid"]), reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame, template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария."""
    if glossary_df.empty or not text:
        return text

    pairs = sorted(zip(glossary_df["source_term"], glossary_df["target_term"]), key=lambda x: len(x[0]), reverse=True)
    protected = []

    def _store(replacement: str) -> str:
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for source_term, target_term in pairs:
        pattern = re.compile(rf"(?<!\w){re.escape(source_term)}(?!\w)", re.IGNORECASE)
        result = pattern.sub(lambda match: _store(template.format(source=match.group(0), target=target_term)), result)

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [ ]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства двух строк на основе расстояния Левенштейна (0–100)."""
    if not a and not b: return 100.0
    longest = max(len(a), len(b))
    dist = edit_distance(a, b) # Функция edit_distance уже есть в Блоке 1
    return round((1.0 - dist / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов (0-100)."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

In [ ]:
import os
from datetime import datetime, timezone

PARALLEL_CORPUS_VARIANT = [
    ("Sarcopenia currently affects 10–27% of people worldwide and 30–37% in the Russian Federation.",
     "В настоящее время распространенность саркопении в мире составляет 10–27%, в Российской Федерации – 30–37%."),
    ("Sarcopenia screening in primary care: opportunities and limitations",
     "Скрининг саркопении на первичном этапе: возможности и ограничения"),
    ("The evolution of sarcopenia diagnostic criteria reflects a shift from muscle mass assessment to an integrative model...",
     "Эволюция диагностических критериев саркопении отражает переход от оценки мышечной массы к интегративной модели...")
]

def build_tmx_for_variant(pairs, path):
    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01", "creationtoolversion": "1.0", "segtype": "sentence",
        "o-tmf": "TMX", "adminlang": "en", "srclang": "en", "datatype": "plaintext", "creationdate": created
    })
    body = ET.SubElement(tmx, "body")
    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = "Фармацевтика и клинические исследования"
        for lang, text in (("en", source), ("ru", target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(ET.tostring(tmx, encoding="unicode"))
    return path

build_tmx_for_variant(PARALLEL_CORPUS_VARIANT, "variant_4.tmx")
print("Файл variant_4.tmx успешно создан!")

Файл variant_4.tmx успешно создан!


## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [ ]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [ ]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("variant_4.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 100, f"Ожидается 100 терминов, получено {len(glossary_df)}"

for column in ["entry_id", "source_term", "target_term", "pos", "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "Sarcopenia"].iloc[0]
assert row["target_term"] == "Саркопения", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "Noun", "Не извлечён termNote[@type='partOfSpeech']"

print("Тест 2 (парсинг моего TBX) — OK")
glossary_df.head()

Тест 2 (парсинг моего TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,Sarcopenia,Саркопения,Noun,A progressive and generalized skeletal mus...,Фармацевтика и клинические исследования
1,tid-002,Screening,Скрининг,Noun,Identification of risk using questionnaire...,Фармацевтика и клинические исследования
2,tid-003,Muscle mass,Мышечная масса,Noun phrase,A component of body composition evaluated ...,Фармацевтика и клинические исследования
3,tid-004,Muscle strength,Мышечная сила,Noun phrase,The primary diagnostic criterion for sarco...,Фармацевтика и клинические исследования
4,tid-005,Handgrip dynamometry,Кистевая динамометрия,Noun phrase,"A simple, inexpensive, and non-invasive me...",Фармацевтика и клинические исследования


In [ ]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("variant_4.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]

assert first["source_segment"].startswith("Sarcopenia currently affects"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("В настоящее время распространенность"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг моего TMX) — OK")
tm_df.head()

Тест 3 (парсинг моего TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,Sarcopenia currently affects 10–27% of peo...,В настоящее время распространенность сарко...,Фармацевтика и клинические исследования,93,
1,2,Sarcopenia screening in primary care: oppo...,Скрининг саркопении на первичном этапе: во...,Фармацевтика и клинические исследования,67,Sarcopenia currently affects 10–27% of peo...
2,3,The evolution of sarcopenia diagnostic cri...,Эволюция диагностических критериев саркопе...,Фармацевтика и клинические исследования,119,Sarcopenia screening in primary care: oppo...


In [ ]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------

exact_query = "Sarcopenia screening in primary care: opportunities and limitations"
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="Sarcopenia currently affects 10–27% of people worldwide and 30–37% in the Russian Federation.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("Sarcopenia screening in hospital care: opportunities and limitations",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("screening sarcopenia",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"

assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"

assert len(calculate_fuzzy_match("sarcopenia", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"

print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [ ]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("Sarcopenia is a disease. Screening is important.",
                             glossary_df)
assert "[Саркопения]" in marked, "Термин из глоссария не подставлен"
assert "[Скрининг]" in marked, "Подставлен не каждый найденный термин"
assert "Sarcopenia" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Muscle mass is important.", glossary_df)
assert "Muscle mass [Мышечная масса]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[Мышечная масса]" not in apply_term_glossary("muscle masses are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"

print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
Sarcopenia [Саркопения] is a disease. Screening [Скрининг] is important.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [ ]:
import os
import csv
import pandas as pd
from datetime import datetime, timezone

# 1. Создаем TMX с 10 предложениями (включая нужные для поиска)
PARALLEL_CORPUS_VARIANT = [
    ("Sarcopenia currently affects 10–27% of people worldwide and 30–37% in the Russian Federation.", "В настоящее время распространенность саркопении в мире составляет 10–27%, в Российской Федерации – 30–37%."),
    ("Sarcopenia screening in primary care: opportunities and limitations", "Скрининг саркопении на первичном этапе: возможности и ограничения"),
    ("The evolution of sarcopenia diagnostic criteria reflects a shift from muscle mass assessment to an integrative model...", "Эволюция диагностических критериев саркопении отражает переход от оценки мышечной массы к интегративной модели..."),
    ("...in which muscle strength serves as the primary criterion and physical performance as an indicator of disease severity.", "...в которых мышечная сила служит первичным критерием, а физическая работоспособность – показателем тяжести."),
    ("Handgrip dynamometry is a simple, inexpensive, and non-invasive method for assessing muscle strength...", "Кистевая динамометрия – простой, недорогой и неинвазивный метод оценки мышечной силы..."),
    ("Assessment of physical performance: functional tests as indicators of sarcopenia severity", "Оценка физической работоспособности: функциональные тесты как показатель тяжести саркопении"),
    ("Bioelectrical impedance analysis (BIA) is an accessible method for assessing body composition based on the electrical resistance of tissues...", "Биоимпедансный анализ (БИА) – доступный метод определения состава тела по электрическому сопротивлению тканей..."),
    ("DXA in whole-body mode is a key method for assessing muscle mass and is based on differential X-ray absorption by soft tissues.", "DXA в режиме «Все тело» – ключевой метод оценки мышечной массы, основанный на поглощении рентгеновского излучения мягкими тканями."),
    ("Handgrip dynamometry cutoff values varied across populations.", "Показана вариабельность пороговых значений кистевой динамометрии между популяциями."),
    ("L.P. Fried et al. (2001) included weakness in the frailty phenotype, considering handgrip strength a marker of vulnerability.", "L.P. Fried и соавт. (2001) включили слабость в фенотип старческой астении, рассматривая результат кистевой динамометрии как маркер уязвимости."),
]

build_tmx_for_variant(PARALLEL_CORPUS_VARIANT, "variant_4.tmx")
tm_df = parse_tmx("variant_4.tmx")

# 2. Запросы для поиска (ICE и High Fuzzy)
QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с контекстом",
        "query": "Sarcopenia screening in primary care: opportunities and limitations",
        "context": "Sarcopenia currently affects 10–27% of people worldwide and 30–37% in the Russian Federation.",
    },
    {
        "label": "2. Fuzzy — заменено 1 слово",
        "query": "...in which muscle strength serves as the main criterion and physical performance as an indicator of disease severity.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The quarterly financial report was approved by the board.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], tm_df, threshold=70.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 70 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)
print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")

if not matches_df.empty:
    display(matches_df[["query", "tuid", "score", "lev_ratio",
                        "token_sort_ratio", "match_type"]].head(6))

# ==============================================================================
# 3. ЭКСПОРТ ФАЙЛОВ В CSV (добавленный блок)
# ==============================================================================
GLOSSARY_CSV = "glossary_smartcat_4.csv"
TM_CSV = "tm_segments_4.csv"
FUZZY_CSV = "fuzzy_report_4.csv"

# Глоссарий
smartcat_df = pd.DataFrame({
    "Source (en)": glossary_df["source_term"],
    "Target (ru)": glossary_df["target_term"],
    "Comment": glossary_df["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)

# Сегменты TMX
tm_df[["tuid", "source_segment", "target_segment", "domain"]].to_csv(
    TM_CSV, index=False, encoding="utf-8-sig")

# Отчёт по найденным совпадениям
if not matches_df.empty:
    matches_df[["query", "tuid", "source_segment", "target_segment",
                "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(
        FUZZY_CSV, index=False, encoding="utf-8-sig")
else:
    # Если вдруг ничего не нашли — создаем пустой файл с заголовками
    pd.DataFrame(columns=["query", "tuid", "source_segment", "target_segment",
                          "score", "lev_ratio", "token_sort_ratio", "match_type"]).to_csv(
        FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in ["variant_4.tbx", "variant_4.tmx", GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    if os.path.exists(name):
        print(f"  {name:<28} {os.path.getsize(name):>7} байт")
    else:
        print(f"  {name:<28} (не найден!)")

print("\nПревью glossary_smartcat_v4.csv:")
display(smartcat_df.head(5))

1. ICE — точное совпадение с контекстом
Запрос: Sarcopenia screening in primary care: opportunities and limitations
  #1 100.00 %  [ICE (101%)  ] tuid=2
      TM src: Sarcopenia screening in primary care: opportunities and limitations
      TM tgt: Скрининг саркопении на первичном этапе: возможности и ограничения
2. Fuzzy — заменено 1 слово
Запрос: ...in which muscle strength serves as the main criterion and physical performance as an indicator of disease severity.
  #1  95.73 %  [High Fuzzy  ] tuid=4
      TM src: ...in which muscle strength serves as the primary criterion and physical performance as an indicator of disease severity.
      TM tgt: ...в которых мышечная сила служит первичным критерием, а физическая работоспособность – показателем тяжести.
3. No match — сегмент вне предметной области
Запрос: The quarterly financial report was approved by the board.
  -> совпадений выше порога 70 % не найдено, сегмент переводится с нуля

Всего кандидатов выше порога: 2


,query,tuid,score,lev_ratio,token_sort_ratio,match_type
0,Sarcopenia screening in primary care: oppo...,2,100.00,100.00,100.00,ICE (101%)
1,...in which muscle strength serves as the ...,4,95.73,95.73,88.89,High Fuzzy



Экспортированные файлы:
  variant_4.tbx                  80794 байт
  variant_4.tmx                   6123 байт
  glossary_smartcat_4.csv        18663 байт
  tm_segments_4.csv               3822 байт
  fuzzy_report_4.csv               853 байт

Превью glossary_smartcat_v4.csv:


,Source (en),Target (ru),Comment
0,Sarcopenia,Саркопения,A progressive and generalized skeletal mus...
1,Screening,Скрининг,Identification of risk using questionnaire...
2,Muscle mass,Мышечная масса,A component of body composition evaluated ...
3,Muscle strength,Мышечная сила,The primary diagnostic criterion for sarco...
4,Handgrip dynamometry,Кистевая динамометрия,"A simple, inexpensive, and non-invasive me..."


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
TBX — это словарь терминов, где написано слово на иностранном языке и его русский эквивалент. Он нужен, чтобы переводчик не путался в сложных медицинских понятиях. А TMX — это база уже готовых переведенных предложений. Она нужна для того, чтобы не переводить одно и то же предложение заново, а просто найти его в памяти.
2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
В файлах XML язык обозначается атрибутом xml:lang, но при чтении файла программа воспринимает его не как просто "lang", а как длинную строку с техническим адресом в начале. Если искать просто "lang", программа ничего не найдет. В коде мы создали специальную переменную XML_LANG, в которой записан этот полный адрес, и теперь программа всегда правильно определяет язык.
3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это компенсирует Token Sort Ratio?
Levenshtein сравнивает текст по буквам и плохо работает, если в предложении переставлены слова. Token Sort Ratio сначала сортирует все слова в предложении по алфавиту, а потом сравнивает. В коде мы берем лучший результат из этих двух проверок, поэтому программа находит совпадения, даже если порядок слов изменен.
4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно для стоимости проекта в Smartcat?
Обычное совпадение (100%) означает, что переведенное предложение совпало с тем, что было в памяти. ICE (101%) означает, что совпало не только само предложение, но и то, которое шло перед ним (контекст). В Smartcat за ICE-совпадения платят меньше или не платят вообще, так как переводчику не нужно проверять контекст и вносить правки, что снижает стоимость проекта.
5. Какие проблемы возникли при автоматической подстановке терминов и как вы их решили?
Программа могла запутаться, если сначала заменяла короткий термин внутри длинного, или заменяла одно и то же слово несколько раз. В коде это решено двумя способами: термины сортируются по убыванию длины (сначала длинные, потом короткие), а уже замененные фрагменты временно скрываются с помощью специальных маркеров, чтобы программа не обрабатывала их повторно.


**Ваши выводы:**
Я сделала программу, которая помогает переводчику работать с медицинскими текстами. Она читает два файла: словарь терминов (TBX), где написано слово на английском и его перевод на русский (термины на тему Фармацевтика и клинические исследования), и память переводов (TMX) — базу уже готовых переведённых предложений. Программа умеет искать в этой базе похожие предложения и показывать процент совпадения. В конце она сохраняет словарь в CSV-файл. Все 5 проверок кода прошли успешно, все файлы созданы и готовы.
_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.